## Lecture 4 -- Parallel Computing - Part 1 

### Excercise 1 -- Monte carlo

In [ ]:
import random
import statistics
import time
from multiprocessing import Pool, cpu_count
 
 
def estimate_pi_serial(num_samples: int) -> float:
    hits = 0
    for _ in range(num_samples):
        x = random.random()
        y = random.random()
        if x * x + y * y <= 1.0:
            hits += 1
    return 4.0 * hits / num_samples
 
 
def timed_run(num_samples: int, n_runs: int = 3):
    
    times: list[float] = []
    estimates: list[float] = []
 
    for run in range(1, n_runs + 1):
        t0 = time.perf_counter()
        pi_est = estimate_pi_serial(num_samples)
        t1 = time.perf_counter()
 
        elapsed = t1 - t0
        times.append(elapsed)
        estimates.append(pi_est)
 
        error = abs(pi_est - 3.141592653589793)
        print(
            f"  Run {run}: π ≈ {pi_est:.6f}  |  "
            f"error = {error:.2e}  |  "
            f"time = {elapsed:.4f} s"
        )
 
    return statistics.median(times), estimates
 
if __name__ == "__main__":
    import math
 
    NUM_SAMPLES = 10_000_000
    N_RUNS = 3
    TRUE_PI = math.pi
 
    print("Monte Carlo")
    
    print(f"  Samples per run : {NUM_SAMPLES:,}")
    print(f"  Number of runs  : {N_RUNS}")
    print(f"  True π          : {TRUE_PI:.10f}")
  
 
    median_time, estimates = timed_run(NUM_SAMPLES, N_RUNS)
 
    print("-" * 60)
 
    mean_est = statistics.mean(estimates)
 

    print(f"  Median time: {median_time:.4f} s  ← serial baseline")
   
   
   
  

Monte Carlo
  Samples per run : 10,000,000
  Number of runs  : 3
  True π          : 3.1415926536
  Run 1: π ≈ 3.142548  |  error = 9.56e-04  |  time = 3.9816 s
  Run 2: π ≈ 3.140604  |  error = 9.88e-04  |  time = 4.1592 s
  Run 3: π ≈ 3.141100  |  error = 4.93e-04  |  time = 3.9507 s
------------------------------------------------------------
  Median time: 3.9816 s  ← serial baseline


### Questions

####  How accurate is the estimate? Run several times — does it vary?
- The estimate is accurate to the 4th decimal.
- Yes, everything after the 4th decimal changes with each run.

####  What is the serial time? This will be your speedup denominator in E3.
- Median time: 3.9373

### Excercise 2 -- Parallel Implementation

In [ ]:
def estimate_pi_chunk(num_samples: int) -> float:
    hits = 0
    for _ in range(num_samples):
        x = random.random()
        y = random.random()
        if x * x + y * y <= 1.0:
            hits += 1
    return hits


from multiprocessing import Pool
import os

def estimate_pi_parallel(num_samples: int, num_processes: int) -> float:
    
    samples_per_worker = num_samples // num_processes

    tasks = [samples_per_worker] * num_processes

    with Pool(processes=num_processes) as pool:
        results = pool.map(estimate_pi_chunk, tasks)

    total_hits = sum(results)
    total_samples = samples_per_worker * num_processes

    return 4.0 * total_hits / total_samples




def benchmark_parallel(num_samples: int):
    import time
    import math
    max_procs = os.cpu_count()

    for p in range(1, max_procs + 1):
        t0 = time.perf_counter()
        pi_est = estimate_pi_parallel(num_samples, p)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        speedup =  3.9373 / elapsed

        print(
            f"Procs: {p:2d} | π ≈ {pi_est:.6f} | "
            f"time = {elapsed:.4f} s | speedup = {speedup:.2f}x"
        )


if __name__ == "__main__":
    NUM_SAMPLES = 10000000

    benchmark_parallel(NUM_SAMPLES)

####  Execution Notes
- Code must be run from a .py file (multiprocessing does not work reliably in notebooks).

---

#### Results

| Processes | π Estimate | Time (s) | Speedup |
|----------:|-----------:|---------:|--------:|
| 1  | 3.141628 | 1.8856 | 2.09x |
| 2  | 3.140458 | 1.0254 | 3.84x |
| 3  | 3.142446 | 0.8212 | 4.79x |
| 4  | 3.140732 | 0.6174 | 6.38x |
| 5  | 3.141858 | 0.5649 | 6.97x |
| 6  | 3.141890 | 0.5171 | 7.61x |
| 7  | 3.141806 | 0.4854 | 8.11x |
| 8  | 3.141906 | 0.4815 | 8.18x |
| 9  | 3.141968 | 0.5039 | 7.81x |
| 10 | 3.141813 | 0.4724 | 8.33x |
| 11 | 3.141960 | 0.4967 | 7.93x |
| 12 | 3.142204 | 0.4958 | 7.94x |
| 13 | 3.141732 | 0.4728 | 8.33x |
| 14 | 3.141989 | 0.4809 | 8.19x |
| 15 | 3.141165 | 0.4965 | 7.93x |
| 16 | 3.142036 | 0.5446 | 7.23x |

---

#### Questions

**Do all worker counts give the same \( \hat{\pi} \)? Why or why not?**

No, they do not give exactly the same value. Each worker generates its own random set of points, so the Monte Carlo estimate varies slightly due to randomness.

---

**At which count do you first see a meaningful speedup?**

At around **3 processes**, where the speedup is approximately **5×**, which is clearly noticeable and meaningful.

### eXCERCISE 3 --- ANALYZE RESULTS

#### Questions to Discuss

**1. At which worker count \( p^* \) is the speedup maximum?**

The maximum observed speedup occurs the first time around 10


---

**2. Does speedup plateau or drop beyond \( p^* \)? Why?**

Yes the speedup slightly drops and plateus

This happens beacuse of several factors such as, IPC, Process creation, limited cores but also as amdahlls law states, the parallel gains are bounded by the serial fraction

**3. Back-solve implied serial fraction**

$$
s = \frac{\frac{1}{S_{p^*}} - \frac{1}{p^*}}{1 - \frac{1}{p^*}}
$$

Using:
- $S_{p^*} = 8.33$
- $p^* = 10$

$$
s = \frac{\frac{1}{8.33} - \frac{1}{10}}{1 - \frac{1}{10}}
\approx \frac{0.120 - 0.100}{0.9}
\approx \frac{0.020}{0.9}
\approx 0.022
$$

So the implied serial fraction is approximately:

👉 $s \approx 0.022$ (≈ 2.2%)